In [ ]:
!apt-get update -qq && apt-get install -y -qq openjdk-11-jdk-headless

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

!pip install -q --upgrade \
    "numpy==1.26.4" \
    "pyspark==3.5.1" \
    "sentence-transformers" \
    "pandas" \
    "pyarrow>=16,<18" \
    "scipy==1.13.1" \
    "scikit-learn==1.5.2"

print("Dependencies ready.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Dependencies ready.


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T


spark = (
    SparkSession.builder
    .appName("CSE488-Laptop-Processing")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

INPUT_PATH = "/content/combined_laptops (2).csv"

df = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(INPUT_PATH)
)

print("Original rows:", df.count())
df.printSchema()

Original rows: 2955
root
 |-- title: string (nullable = true)
 |-- price_usd: string (nullable = true)
 |-- price_original: string (nullable = true)
 |-- price_original_currency: string (nullable = true)
 |-- cpu: string (nullable = true)
 |-- ram_gb: string (nullable = true)
 |-- storage: string (nullable = true)
 |-- gpu: string (nullable = true)
 |-- display: string (nullable = true)
 |-- battery: string (nullable = true)
 |-- category: string (nullable = true)
 |-- document_text: string (nullable = true)
 |-- has_review_text: string (nullable = true)
 |-- source_dataset: string (nullable = true)
 |-- source_collection_date: string (nullable = true)
 |-- source_row_id: string (nullable = true)
 |-- has_page_provenance: string (nullable = true)



In [4]:
print("Missing values before deduplication:")

missing_before = (
    clean
    .select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in [
            "price_usd",
            "cpu",
            "ram_gb",
            "storage",
            "gpu",
            "display",
            "battery",
            "category"
        ]
    ])
)

missing_before.show(truncate=False)

Missing values before deduplication:
+---------+---+------+-------+---+-------+-------+--------+
|price_usd|cpu|ram_gb|storage|gpu|display|battery|category|
+---------+---+------+-------+---+-------+-------+--------+
|0        |43 |26    |21     |171|0      |1720   |3       |
+---------+---+------+-------+---+-------+-------+--------+



In [5]:
from pyspark.ml.feature import (
    RegexTokenizer,
    HashingTF,
    MinHashLSH
)


tokenizer = RegexTokenizer(
    inputCol="document_text",
    outputCol="tokens",
    pattern=r"\W+",
    minTokenLength=2
)

tokenized = tokenizer.transform(clean)


hashing_tf = HashingTF(
    inputCol="tokens",
    outputCol="tf_vec",
    numFeatures=1 << 14,
    binary=True
)

featurized = hashing_tf.transform(tokenized)


mh = MinHashLSH(
    inputCol="tf_vec",
    outputCol="minhash",
    numHashTables=5
)

mh_model = mh.fit(featurized)

hashed = mh_model.transform(featurized)


JACCARD_DIST_THRESHOLD = 0.2


pairs = (
    mh_model
    .approxSimilarityJoin(
        hashed,
        hashed,
        JACCARD_DIST_THRESHOLD,
        distCol="jaccard_dist"
    )
    .filter(
        F.col("datasetA.row_uid")
        < F.col("datasetB.row_uid")
    )
    .select(
        F.col("datasetA.row_uid").alias("uid_a"),
        F.col("datasetB.row_uid").alias("uid_b"),
        "jaccard_dist"
    )
)

print(
    "Near-duplicate pairs found:",
    pairs.count()
)

pairs.orderBy("jaccard_dist").show(
    10,
    truncate=60
)

Near-duplicate pairs found: 31064
+--------------------------------+--------------------------------+------------+
|                           uid_a|                           uid_b|jaccard_dist|
+--------------------------------+--------------------------------+------------+
|8c05fcb3b6045a1d590d02f6417c1693|95bda8cea4a3dcd1b0c3c55a42f225c2|         0.0|
|50e06f46fafb3ba7be29ab6cab7cd26c|8e7b6aeb3007979d8f975671d818ea88|         0.0|
|9b298bcef46e25021005a702781cf06d|fb5102ec4ef60519d846420eaef3bdcf|         0.0|
|a08f38fb7be016f1dcbc314ca45276e4|bb8fcf8d9b5e4949331e77cf93f7adc2|         0.0|
|67d5d7e67a13caa1ec3a15dbccd14d7a|df9ee4894493796a1abc2e6923b14c04|         0.0|
|0414d6e5cd5cd697998f983e82cd2d8f|c06aa43dac10c49642100fd347a6e282|         0.0|
|6856c6f63272d1a6d6eeca1a7d2cf1de|c0fd5e58dab70ebba03a4bc348d7a7b8|         0.0|
|39d6f02d79e29fce99a7d354dad12e71|bb8fcf8d9b5e4949331e77cf93f7adc2|         0.0|
|99315a9b6e952242a4c3a0479e9ac851|ddd8f18988ce8c48346270a73dadd5ec|        

In [ ]:
import pandas as pd


raw_edge_pdf = pairs.select("uid_a", "uid_b").toPandas()

raw_degree = pd.concat([raw_edge_pdf["uid_a"], raw_edge_pdf["uid_b"]]).value_counts()

n_rows = clean.count()

print("Rows with at least one near-duplicate match:", len(raw_degree), f"of {n_rows} total rows")
print("Average matches per matched row:", round(raw_degree.mean(), 2))
print("\nMatch-count distribution (per row):")
print(raw_degree.describe())

Rows with at least one near-duplicate match: 1708 of 2955 total rows
Average matches per matched row: 36.37

Match-count distribution (per row):
count    1708.000000
mean       36.374707
std        65.247015
min         1.000000
25%         1.000000
50%         3.000000
75%        25.000000
max       197.000000
Name: count, dtype: float64


In [ ]:
PRICE_TOLERANCE = 0.15  

spec_a = hashed.select(
    F.col("row_uid").alias("uid_a"),
    F.col("ram_gb").alias("ram_a"),
    F.col("price_usd").alias("price_a"),
)
spec_b = hashed.select(
    F.col("row_uid").alias("uid_b"),
    F.col("ram_gb").alias("ram_b"),
    F.col("price_usd").alias("price_b"),
)

pairs_with_specs = (
    pairs
    .join(spec_a, on="uid_a", how="left")
    .join(spec_b, on="uid_b", how="left")
)

pairs_filtered = (
    pairs_with_specs
    .withColumn(
        "ram_compatible",
        F.col("ram_a").isNull() | F.col("ram_b").isNull()
        | (F.col("ram_a") == F.col("ram_b"))
    )
    .withColumn(
        "price_compatible",
        F.col("price_a").isNull() | F.col("price_b").isNull()
        | (
            F.abs(F.col("price_a") - F.col("price_b"))
            / F.greatest(F.col("price_a"), F.col("price_b"), F.lit(1.0))
            <= PRICE_TOLERANCE
        )
    )
    .filter(F.col("ram_compatible") & F.col("price_compatible"))
    .select("uid_a", "uid_b", "jaccard_dist")
    .cache()
)

raw_pair_count = pairs.count()
filtered_pair_count = pairs_filtered.count()

print("Candidate pairs before spec-agreement guard:", raw_pair_count)
print("Candidate pairs after spec-agreement guard :", filtered_pair_count)
print(
    "Pairs rejected for RAM/price conflict       :",
    raw_pair_count - filtered_pair_count
)


Candidate pairs before spec-agreement guard: 31064
Candidate pairs after spec-agreement guard : 3698
Pairs rejected for RAM/price conflict       : 27366


In [ ]:

class UnionFind:
    def __init__(self):
        self.parent = {}

    def find(self, x):
        self.parent.setdefault(x, x)
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:

            if ra < rb:
                self.parent[rb] = ra
            else:
                self.parent[ra] = rb


edge_pdf = pairs_filtered.select("uid_a", "uid_b").toPandas()

uf = UnionFind()
for a, b in zip(edge_pdf["uid_a"], edge_pdf["uid_b"]):
    uf.union(a, b)

matched_uids = set(edge_pdf["uid_a"]) | set(edge_pdf["uid_b"])
cluster_map = {uid: uf.find(uid) for uid in matched_uids}

cluster_pdf = pd.DataFrame({
    "row_uid": list(cluster_map.keys()),
    "cluster_id": list(cluster_map.values()),
})

cluster_sizes = cluster_pdf["cluster_id"].value_counts()

print("Duplicate clusters found:", len(cluster_sizes))
print("Rows involved in a duplicate cluster:", len(cluster_pdf))
print("\nCluster size distribution:")
print(cluster_sizes.describe())
print("\nCluster size histogram (cluster size -> number of clusters that size):")
print(cluster_sizes.value_counts().sort_index())


Duplicate clusters found: 334
Rows involved in a duplicate cluster: 1329

Cluster size distribution:
count    334.000000
mean       3.979042
std        5.669277
min        2.000000
25%        2.000000
50%        2.000000
75%        3.000000
max       52.000000
Name: count, dtype: float64

Cluster size histogram (cluster size -> number of clusters that size):
count
2     201
3      54
4      27
5       5
6       5
7       3
8      15
9       5
10      5
11      1
12      2
16      2
17      1
19      1
24      1
29      1
36      1
40      2
42      1
52      1
Name: count, dtype: int64


In [ ]:
to_drop_pdf = cluster_pdf.loc[cluster_pdf["row_uid"] != cluster_pdf["cluster_id"], ["row_uid"]]

to_drop = spark.createDataFrame(to_drop_pdf)

deduped = (
    hashed
    .join(to_drop, on="row_uid", how="left_anti")
    .drop("tokens", "tf_vec", "minhash")
    .cache()
)

original_count = clean.count()
deduped_count = deduped.count()

print("Rows before deduplication:", original_count)
print("Rows after deduplication :", deduped_count)
print("Duplicate rows removed   :", original_count - deduped_count)
print(
    "(computed via connected components on the spec-filtered similarity "
    "graph, not a naive pairwise rule -- see Step 5 methodology)"
)


Rows before deduplication: 2955
Rows after deduplication : 1960
Duplicate rows removed   : 995
(computed via connected components on the spec-filtered similarity graph, not a naive pairwise rule -- see Step 5 methodology)


In [ ]:
N_CLUSTERS_TO_INSPECT = 5

largest_clusters = cluster_sizes.head(N_CLUSTERS_TO_INSPECT).index.tolist()

cluster_id_lookup = spark.createDataFrame(cluster_pdf)

spot_check = (
    hashed
    .select("row_uid", "title", "cpu", "ram_gb", "price_usd", "category")
    .join(cluster_id_lookup, on="row_uid", how="inner")
    .filter(F.col("cluster_id").isin(largest_clusters))
    .orderBy("cluster_id")
    .toPandas()
)

print(f"Members of the {N_CLUSTERS_TO_INSPECT} largest duplicate clusters "
      f"(inspect for false-positive merges):")
display(spot_check)


Members of the 5 largest duplicate clusters (inspect for false-positive merges):


,row_uid,title,cpu,ram_gb,price_usd,category,cluster_id
0,04a0f8a0aa06d32e0b5da2c0ba448ed3,"Apple 16"" MacBook Pro (M5 Max, Nano-Texture Gl...",Apple M5 Max,64.0,5549.0,Creative Laptops,00027776a020cbf5d6338869c4a90068
1,fe54f0cc4fa844d8ed44d711af9c9966,"Apple 16"" MacBook Pro (M5 Pro, Nano-Texture Gl...",Apple M5 Pro,64.0,3799.0,Creative Laptops,00027776a020cbf5d6338869c4a90068
2,00027776a020cbf5d6338869c4a90068,"Apple 14"" MacBook Pro (M5 Max, Nano-Texture Gl...",Apple M5 Max,64.0,6249.0,Creative Laptops,00027776a020cbf5d6338869c4a90068
3,db71d04a56719abc2a59547429d5a89b,"Apple 14"" MacBook Pro (M5 Pro, Nano-Texture Gl...",Apple M5 Pro,64.0,3999.0,Creative Laptops,00027776a020cbf5d6338869c4a90068
4,f4f0e51b73f88c470d099fcabc685c59,"Apple 14"" MacBook Pro (M5 Max, Silver)",Apple M5 Max,64.0,4699.0,Creative Laptops,00027776a020cbf5d6338869c4a90068
...,...,...,...,...,...,...,...
205,d99babfaab8fb057ef49fd51799116c8,"Apple 16"" MacBook Pro (M5 Pro, Space Black)",Apple M5 Pro,48.0,5099.0,Creative Laptops,0f583164d1e9ce3c8e5e89a1c73ef342
206,84d657fd7f9cd1d0cdc86b4308571dd3,"Apple 16"" MacBook Pro (M5 Pro, Nano-Texture Gl...",Apple M5 Pro,48.0,3749.0,Creative Laptops,0f583164d1e9ce3c8e5e89a1c73ef342
207,fd2f4c188ae4392c82a5bc15c10910f7,"Apple 14"" MacBook Pro (M5 Pro, Space Black)",Apple M5 Pro,48.0,3599.0,Creative Laptops,0f583164d1e9ce3c8e5e89a1c73ef342
208,37f25c8f896c5094fbfd39e45d65a056,"Apple 16"" MacBook Pro (M5 Max, Nano-Texture Gl...",Apple M5 Max,48.0,5149.0,Creative Laptops,0f583164d1e9ce3c8e5e89a1c73ef342


In [12]:
DEDUP_CSV = "/content/deduplicated_laptops.csv"

(
    deduped
    .toPandas()
    .to_csv(
        DEDUP_CSV,
        index=False
    )
)

print("Saved:", DEDUP_CSV)

Saved: /content/deduplicated_laptops.csv


In [13]:
!pip install -q --upgrade \
    "numpy==1.26.4" \
    "scipy==1.13.1" \
    "scikit-learn==1.5.2"

In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, f1_score
IMPUTATION_LOG = []


In [15]:
product_df = deduped.toPandas()

print("Product-level dataframe:", product_df.shape)

Product-level dataframe: (1960, 18)


In [ ]:
def parse_battery_wh(value):
    if pd.isna(value):
        return np.nan

    text = str(value)

    match = re.search(
        r"(\d+(?:\.\d+)?)\s*Wh",
        text,
        re.IGNORECASE
    )

    if match:
        return float(match.group(1))

    return np.nan


product_df["battery_wh"] = (
    product_df["battery"]
    .apply(parse_battery_wh)
)

In [17]:
def parse_display_inches(value):
    if pd.isna(value):
        return np.nan

    text = str(value)

    match = re.search(
        r"(\d+(?:\.\d+)?)\s*(?:inch|inches|\"|”)",
        text,
        re.IGNORECASE
    )

    if match:
        return float(match.group(1))

    return np.nan


product_df["display_inches"] = (
    product_df["display"]
    .apply(parse_display_inches)
)

In [ ]:
MIN_KNOWN_FOR_MODEL = 20

MIN_HOLDOUT_FOR_METRICS = 5

VALIDATION_FRACTION = 0.2


def _confidence_label(is_numeric_target, metric_value):

    if metric_value is None:
        return "NOT VALIDATED (holdout too small)"

    if is_numeric_target:
        if metric_value >= 0.6:
            return "HIGH"
        elif metric_value >= 0.3:
            return "MODERATE"
        else:
            return "LOW"
    else:
        if metric_value >= 0.7:
            return "HIGH"
        elif metric_value >= 0.4:
            return "MODERATE"
        else:
            return "LOW"


def regression_impute(
    data,
    target,
    features,
    model_name
):


    known_mask = data[target].notna()
    missing_mask = data[target].isna()

    known_count = int(known_mask.sum())
    missing_count = int(missing_mask.sum())
    total_count = known_count + missing_count

    print("\n" + "=" * 60)
    print(f"Imputation: {target} ({model_name})")
    print("=" * 60)
    print("Known values  :", known_count)
    print("Missing values:", missing_count, f"({missing_count / total_count:.1%} of column)")

    data[f"{target}_is_imputed"] = False

    if missing_count == 0:
        print("No imputation required.")
        IMPUTATION_LOG.append({
            "target": target, "model_name": model_name,
            "known_count": known_count, "missing_count": missing_count,
            "missing_pct": 0.0, "is_numeric": None,
            "metric_name": None, "metric_value": None,
            "secondary_metric_name": None, "secondary_metric_value": None,
            "confidence": "N/A - nothing missing", "filled_count": 0,
            "status": "skipped_no_missing"
        })
        return data

    if known_count < MIN_KNOWN_FOR_MODEL:
        print(
            f"WARNING: Only {known_count} known values "
            f"(< {MIN_KNOWN_FOR_MODEL} minimum) -- too few to train a "
            f"reliable model. Leaving {missing_count} values missing."
        )
        IMPUTATION_LOG.append({
            "target": target, "model_name": model_name,
            "known_count": known_count, "missing_count": missing_count,
            "missing_pct": round(missing_count / total_count, 3), "is_numeric": None,
            "metric_name": None, "metric_value": None,
            "secondary_metric_name": None, "secondary_metric_value": None,
            "confidence": "NOT ATTEMPTED - insufficient known values",
            "filled_count": 0, "status": "skipped_insufficient_data"
        })
        return data

    if missing_count / total_count > 0.7:
        print(
            f"WARNING: {missing_count / total_count:.0%} of \'{target}\' is "
            f"missing. The model is trained on a small, possibly "
            f"unrepresentative slice of the data -- treat imputed values "
            f"with extra caution."
        )

    X = data[features].copy()
    y = data[target].copy()

    X_known = X.loc[known_mask]
    y_known = y.loc[known_mask]
    X_missing = X.loc[missing_mask]

    is_numeric_target = pd.api.types.is_numeric_dtype(y_known)

    y_categories = None
    if not is_numeric_target:
        y_known_encoded, y_categories = pd.factorize(y_known)
        y_known = pd.Series(y_known_encoded, index=y_known.index, name=target)

    categorical_features = [
        c for c in features
        if not pd.api.types.is_numeric_dtype(X[c])
    ]
    numerical_features = [
        c for c in features
        if pd.api.types.is_numeric_dtype(X[c])
    ]

    print("Numeric features    :", numerical_features)
    print("Categorical features:", categorical_features)

    if not numerical_features and not categorical_features:
        print("WARNING: No usable features available for imputation.")
        IMPUTATION_LOG.append({
            "target": target, "model_name": model_name,
            "known_count": known_count, "missing_count": missing_count,
            "missing_pct": round(missing_count / total_count, 3), "is_numeric": is_numeric_target,
            "metric_name": None, "metric_value": None,
            "secondary_metric_name": None, "secondary_metric_value": None,
            "confidence": "NOT ATTEMPTED - no usable features",
            "filled_count": 0, "status": "skipped_no_features"
        })
        return data

    def build_pipeline():
        transformers = []

        if numerical_features:
            numeric_pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ])
            transformers.append(("num", numeric_pipeline, numerical_features))

        if categorical_features:
            categorical_pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ])
            transformers.append(("cat", categorical_pipeline, categorical_features))

        preprocessor = ColumnTransformer(transformers=transformers)

        if is_numeric_target:
            model = RandomForestRegressor(
                n_estimators=300, max_depth=15,
                min_samples_leaf=2, random_state=42, n_jobs=-1
            )
        else:
            model = RandomForestClassifier(
                n_estimators=300, max_depth=15,
                min_samples_leaf=2, random_state=42, n_jobs=-1
            )

        return Pipeline([("preprocessor", preprocessor), ("model", model)])

    metric_name = metric_value = None
    secondary_metric_name = secondary_metric_value = None

    can_stratify = (not is_numeric_target) and (y_known.value_counts().min() >= 2)
    holdout_size = int(round(known_count * VALIDATION_FRACTION))

    if holdout_size >= MIN_HOLDOUT_FOR_METRICS:
        try:
            X_tr, X_val, y_tr, y_val = train_test_split(
                X_known, y_known,
                test_size=VALIDATION_FRACTION,
                random_state=42,
                stratify=y_known if can_stratify else None
            )

            val_pipeline = build_pipeline()
            val_pipeline.fit(X_tr, y_tr)
            val_preds = val_pipeline.predict(X_val)

            if is_numeric_target:
                metric_name, metric_value = "R2", round(float(r2_score(y_val, val_preds)), 3)
                secondary_metric_name = "MAE"
                secondary_metric_value = round(float(mean_absolute_error(y_val, val_preds)), 3)
            else:
                metric_name, metric_value = "Accuracy", round(float(accuracy_score(y_val, val_preds)), 3)
                secondary_metric_name = "Macro-F1"
                secondary_metric_value = round(
                    float(f1_score(y_val, val_preds, average="macro", zero_division=0)), 3
                )

            confidence = _confidence_label(is_numeric_target, metric_value)

            print(
                f"Validation ({len(X_val)} held-out rows) -- "
                f"{metric_name}: {metric_value}, {secondary_metric_name}: {secondary_metric_value} "
                f"-> confidence: {confidence}"
            )

        except Exception as exc:
            print(f"NOTE: validation split failed ({exc}); skipping metric reporting.")
            confidence = _confidence_label(is_numeric_target, None)
    else:
        print(
            f"NOTE: only {known_count} known values -> holdout would be "
            f"{holdout_size} rows, too small to report a meaningful metric. "
            f"Imputing without a validation score; treat as unverified."
        )
        confidence = _confidence_label(is_numeric_target, None)

    final_pipeline = build_pipeline()
    final_pipeline.fit(X_known, y_known)
    predictions = final_pipeline.predict(X_missing)

    if is_numeric_target:
        predictions = np.asarray(predictions, dtype=float)
        predictions = np.where(np.isfinite(predictions), predictions, np.nan)
    else:
        predictions = y_categories[predictions]
        predictions = np.asarray(predictions, dtype=object)

    data.loc[missing_mask, target] = predictions

    filled_mask = missing_mask & data[target].notna()
    data.loc[filled_mask, f"{target}_is_imputed"] = True

    valid_predictions = predictions[~pd.isna(predictions)]
    print(f"Filled {len(valid_predictions)} missing {target} values.")

    if len(valid_predictions) and is_numeric_target:
        print(
            "Prediction range:",
            round(float(valid_predictions.min()), 2), "to",
            round(float(valid_predictions.max()), 2)
        )
    elif len(valid_predictions) and not is_numeric_target:
        print(
            "Predicted categories (top 5):",
            pd.Series(valid_predictions).value_counts().head(5).index.tolist()
        )

    IMPUTATION_LOG.append({
        "target": target, "model_name": model_name,
        "known_count": known_count, "missing_count": missing_count,
        "missing_pct": round(missing_count / total_count, 3),
        "is_numeric": is_numeric_target,
        "metric_name": metric_name, "metric_value": metric_value,
        "secondary_metric_name": secondary_metric_name,
        "secondary_metric_value": secondary_metric_value,
        "confidence": confidence,
        "filled_count": int(len(valid_predictions)),
        "status": "imputed"
    })

    return data


In [ ]:
text_columns = [
    "cpu", "storage", "gpu", "category"
]
for col in text_columns:
    if col in product_df.columns:
        product_df[col] = product_df[col].astype("string")

for col in ["price_usd", "ram_gb", "battery_wh", "display_inches"]:
    if col in product_df.columns:
        product_df[col] = pd.to_numeric(product_df[col], errors="coerce")


In [ ]:
ram_features = [
    "price_usd",
    "cpu",
    "storage",
    "gpu",
    "display_inches",
    "battery_wh",
    "category"
]

for col in text_columns:
    if col in product_df.columns:
        product_df[col] = product_df[col].astype(object)
        product_df.loc[product_df[col].isna(), col] = None

product_df = regression_impute(
    product_df,
    target="ram_gb",
    features=ram_features,
    model_name="RAM regression"
)


Imputation: ram_gb (RAM regression)
Known values  : 1935
Missing values: 25 (1.3% of column)
Numeric features    : ['price_usd', 'display_inches', 'battery_wh']
Categorical features: ['cpu', 'storage', 'gpu', 'category']
Validation (387 held-out rows) -- R2: 0.642, MAE: 4.826 -> confidence: HIGH
Filled 25 missing ram_gb values.
Prediction range: 12.51 to 73.66


In [21]:
battery_features = [
    "price_usd",
    "cpu",
    "ram_gb",
    "storage",
    "gpu",
    "display_inches",
    "category"
]

product_df = regression_impute(
    product_df,
    target="battery_wh",
    features=battery_features,
    model_name="Battery regression"
)


Imputation: battery_wh (Battery regression)
Known values  : 319
Missing values: 1641 (83.7% of column)
Numeric features    : ['price_usd', 'ram_gb', 'display_inches']
Categorical features: ['cpu', 'storage', 'gpu', 'category']
Validation (64 held-out rows) -- R2: 0.773, MAE: 6.273 -> confidence: HIGH
Filled 1641 missing battery_wh values.
Prediction range: 38.48 to 98.92


In [22]:
display_features = [
    "price_usd",
    "cpu",
    "ram_gb",
    "storage",
    "gpu",
    "battery_wh",
    "category"
]

product_df = regression_impute(
    product_df,
    target="display_inches",
    features=display_features,
    model_name="Display regression"
)


Imputation: display_inches (Display regression)
Known values  : 1947
Missing values: 13 (0.7% of column)
Numeric features    : ['price_usd', 'ram_gb', 'battery_wh']
Categorical features: ['cpu', 'storage', 'gpu', 'category']
Validation (390 held-out rows) -- R2: 0.677, MAE: 0.488 -> confidence: HIGH
Filled 13 missing display_inches values.
Prediction range: 13.96 to 16.31


In [ ]:

battery_missing = (
    product_df["battery"].isna()
    & product_df["battery_wh"].notna()
)

product_df.loc[
    battery_missing,
    "battery"
] = (
    product_df.loc[
        battery_missing,
        "battery_wh"
    ]
    .round(1)
    .astype(str)
    + " Wh"
)

product_df["battery_is_imputed"] = battery_missing



display_missing = (
    product_df["display"].isna()
    & product_df["display_inches"].notna()
)

product_df.loc[
    display_missing,
    "display"
] = (
    product_df.loc[
        display_missing,
        "display_inches"
    ]
    .round(1)
    .astype(str)
    + " inch"
)

product_df["display_is_imputed"] = display_missing


In [ ]:
cpu_features = [
    "price_usd",
    "ram_gb",
    "storage",
    "gpu",
    "display",
    "battery",
    "category"
]

for col in ['display', 'battery']:
    if col in product_df.columns:
        product_df[col] = product_df[col].astype(object)
        product_df.loc[product_df[col].isna(), col] = None


product_df = regression_impute(
    product_df,
    target="cpu",
    features=cpu_features,
    model_name="CPU regression"
)


Imputation: cpu (CPU regression)
Known values  : 1921
Missing values: 39 (2.0% of column)
Numeric features    : ['price_usd', 'ram_gb']
Categorical features: ['storage', 'gpu', 'display', 'battery', 'category']
Validation (385 held-out rows) -- Accuracy: 0.343, Macro-F1: 0.107 -> confidence: LOW
Filled 39 missing cpu values.
Predicted categories (top 5): ['Intel Core i7 11th Gen', 'Intel Core i5 7200U 2.5GHz', 'Intel Core Ultra 7 (Series 2)', 'Intel Core Ultra 7 366H', 'Intel Core i7 7500U 2.7GHz']


In [25]:
storage_features = [
    "price_usd",
    "cpu",
    "ram_gb",
    "gpu",
    "display",
    "battery",
    "category"
]

product_df = regression_impute(
    product_df,
    target="storage",
    features=storage_features,
    model_name="Storage regression"
)


Imputation: storage (Storage regression)
Known values  : 1942
Missing values: 18 (0.9% of column)
Numeric features    : ['price_usd', 'ram_gb']
Categorical features: ['cpu', 'gpu', 'display', 'battery', 'category']
Validation (389 held-out rows) -- Accuracy: 0.36, Macro-F1: 0.099 -> confidence: LOW
Filled 18 missing storage values.
Predicted categories (top 5): ['512GB', '512 GB', '256GB SSD', '1 TB']


In [26]:
gpu_features = [
    "price_usd",
    "cpu",
    "ram_gb",
    "storage",
    "display",
    "battery",
    "category"
]

product_df = regression_impute(
    product_df,
    target="gpu",
    features=gpu_features,
    model_name="GPU regression"
)


Imputation: gpu (GPU regression)
Known values  : 1807
Missing values: 153 (7.8% of column)
Numeric features    : ['price_usd', 'ram_gb']
Categorical features: ['cpu', 'storage', 'display', 'battery', 'category']
Validation (362 held-out rows) -- Accuracy: 0.403, Macro-F1: 0.067 -> confidence: MODERATE
Filled 153 missing gpu values.
Predicted categories (top 5): ['Intel Iris Xe Graphics', 'Intel Graphics', 'Intel UHD Graphics', 'Intel HD Graphics 620', 'AMD Radeon Graphics']


In [27]:
category_features = [
    "price_usd",
    "cpu",
    "ram_gb",
    "storage",
    "gpu",
    "display",
    "battery"
]

product_df = regression_impute(
    product_df,
    target="category",
    features=category_features,
    model_name="Category regression"
)


Imputation: category (Category regression)
Known values  : 1958
Missing values: 2 (0.1% of column)
Numeric features    : ['price_usd', 'ram_gb']
Categorical features: ['cpu', 'storage', 'gpu', 'display', 'battery']
Validation (392 held-out rows) -- Accuracy: 0.569, Macro-F1: 0.186 -> confidence: MODERATE
Filled 2 missing category values.
Predicted categories (top 5): ['Work Laptops']


In [28]:
imputation_report = pd.DataFrame(IMPUTATION_LOG)

pd.set_option("display.max_colwidth", 40)

print("=" * 100)
print("IMPUTATION QUALITY REPORT")
print("=" * 100)
print(
    "Metrics below come from an 80/20 held-out validation split (see Step 7 "
    "methodology) -- an honest estimate of generalization, not a "
    "training-set score. 'confidence' is a plain-language label derived "
    "from the metric (see _confidence_label() thresholds in Step 7)."
)

report_cols = [
    "target", "model_name", "known_count", "missing_count", "missing_pct",
    "metric_name", "metric_value", "secondary_metric_name",
    "secondary_metric_value", "confidence", "filled_count", "status"
]
imputation_report = imputation_report[report_cols]
display(imputation_report)

IMPUTATION_REPORT_CSV = "/content/imputation_quality_report.csv"
imputation_report.to_csv(IMPUTATION_REPORT_CSV, index=False)
print("\nSaved:", IMPUTATION_REPORT_CSV)


IMPUTATION QUALITY REPORT
Metrics below come from an 80/20 held-out validation split (see Step 7 methodology) -- an honest estimate of generalization, not a training-set score. 'confidence' is a plain-language label derived from the metric (see _confidence_label() thresholds in Step 7).


,target,model_name,known_count,missing_count,missing_pct,metric_name,metric_value,secondary_metric_name,secondary_metric_value,confidence,filled_count,status
0,ram_gb,RAM regression,1935,25,0.013,R2,0.642,MAE,4.826,HIGH,25,imputed
1,battery_wh,Battery regression,319,1641,0.837,R2,0.773,MAE,6.273,HIGH,1641,imputed
2,display_inches,Display regression,1947,13,0.007,R2,0.677,MAE,0.488,HIGH,13,imputed
3,cpu,CPU regression,1921,39,0.020,Accuracy,0.343,Macro-F1,0.107,LOW,39,imputed
4,storage,Storage regression,1942,18,0.009,Accuracy,0.360,Macro-F1,0.099,LOW,18,imputed
5,gpu,GPU regression,1807,153,0.078,Accuracy,0.403,Macro-F1,0.067,MODERATE,153,imputed
6,category,Category regression,1958,2,0.001,Accuracy,0.569,Macro-F1,0.186,MODERATE,2,imputed



Saved: /content/imputation_quality_report.csv


In [ ]:
final_columns = [
    "price_usd",
    "cpu",
    "ram_gb",
    "storage",
    "gpu",
    "display",
    "battery",
    "category"
]

imputed_flag_map = {
    "ram_gb": "ram_gb_is_imputed",
    "battery": "battery_is_imputed",
    "display": "display_is_imputed",
    "cpu": "cpu_is_imputed",
    "storage": "storage_is_imputed",
    "gpu": "gpu_is_imputed",
    "category": "category_is_imputed",
}

print("=" * 70)
print("FINAL MISSING-VALUE REPORT AFTER ALL IMPUTATIONS")
print("=" * 70)

for col in final_columns:

    missing = product_df[col].isna().sum()
    total = len(product_df)
    flag_col = imputed_flag_map.get(col)
    imputed_count = int(product_df[flag_col].sum()) if flag_col in product_df.columns else 0

    print(
        f"{col:25s}: "
        f"{missing:5d}/{total} ({missing / total:.1%}) still missing   |   "
        f"{imputed_count:5d} filled by model"
    )


FINAL MISSING-VALUE REPORT AFTER ALL IMPUTATIONS
price_usd                :     0/1960 (0.0%) still missing   |       0 filled by model
cpu                      :     0/1960 (0.0%) still missing   |      39 filled by model
ram_gb                   :     0/1960 (0.0%) still missing   |      25 filled by model
storage                  :     0/1960 (0.0%) still missing   |      18 filled by model
gpu                      :     0/1960 (0.0%) still missing   |     153 filled by model
display                  :     0/1960 (0.0%) still missing   |       0 filled by model
battery                  :     0/1960 (0.0%) still missing   |    1378 filled by model
category                 :     0/1960 (0.0%) still missing   |       2 filled by model


In [ ]:
product_df = product_df.drop(
    columns=[
        "battery_wh",
        "display_inches",
        "battery_wh_is_imputed",
        "display_inches_is_imputed",
    ],
    errors="ignore"
)


In [31]:
FINAL_CSV = "/content/final_clean_filled_laptops.csv"

product_df.to_csv(
    FINAL_CSV,
    index=False
)

print("=" * 60)
print("FINAL PRODUCT DATASET")
print("=" * 60)

print("Saved:", FINAL_CSV)
print("Rows:", len(product_df))
print("Columns:", len(product_df.columns))

imputed_flag_cols = [c for c in product_df.columns if c.endswith("_is_imputed")]
print(
    "\nEvery imputed value is traceable via these boolean columns:",
    imputed_flag_cols
)

print("\nFirst 5 rows:")
display(product_df.head())


FINAL PRODUCT DATASET
Saved: /content/final_clean_filled_laptops.csv
Rows: 1960
Columns: 25

Every imputed value is traceable via these boolean columns: ['ram_gb_is_imputed', 'battery_is_imputed', 'display_is_imputed', 'cpu_is_imputed', 'storage_is_imputed', 'gpu_is_imputed', 'category_is_imputed']

First 5 rows:


,row_uid,title,price_usd,price_original,price_original_currency,cpu,ram_gb,storage,gpu,display,...,source_collection_date,source_row_id,has_page_provenance,ram_gb_is_imputed,battery_is_imputed,display_is_imputed,cpu_is_imputed,storage_is_imputed,gpu_is_imputed,category_is_imputed
0,558389f6752bb2a5f024a76c8b26a6ed,"ASUS Vivobook S 16"" 3K OLED Intel Co...",1249.99,1249.99,USD,Intel Core Ultra 7 (Series 2),32.0,1TB PCIe,Intel Arc Graphics,"16""",...,2026-08-12,https://www.newegg.com/asus-vivobook...,True,False,False,False,False,False,False,False
1,8e62c8ad9f495809551c090e6e477543,"Acer Aspire Go 15 15.6"" AMD Ryzen 7 ...",714.99,714.99,USD,AMD Ryzen 7 5000 Series,48.0,512GB PCIe Gen4,AMD Radeon Graphics,"15.6""",...,2026-08-12,https://www.newegg.com/acer-america-...,True,False,False,False,False,False,False,False
2,34950ed249e57741546384aef2b1f675,"Acer Aspire Go 15 15.6"" AMD Ryzen 7 ...",449.99,449.99,USD,AMD Ryzen 7 5000 Series,32.0,512GB PCIe Gen4,AMD Radeon Graphics,"15.6""",...,2026-08-12,https://www.newegg.com/acer-america-...,True,False,False,False,False,False,False,False
3,4ac36e517409cb322edb2986286db803,"Lenovo IdeaPad Slim 3 Laptop 15.3"" F...",729.00,729.00,USD,AMD Ryzen 7,16.0,512 GB,AMD Radeon 780M,"15.3""",...,2026-08-12,https://www.newegg.com/lenovo-15-3-f...,True,False,False,False,False,False,False,False
4,87bebf27f4bda45ea60a507532cd3da5,"Acer Aspire Go 15 15.6"" - Intel Core...",899.99,899.99,USD,Intel Core i9 13th Gen,48.0,512GB PCIe,Intel Iris Xe Graphics,"15.6""",...,2026-08-12,https://www.newegg.com/acer-america-...,True,False,False,False,False,False,False,False


In [32]:
final_spark = spark.createDataFrame(product_df)

print(
    "Final Spark product rows:",
    final_spark.count()
)

final_spark.printSchema()

Final Spark product rows: 1960
root
 |-- row_uid: string (nullable = true)
 |-- title: string (nullable = true)
 |-- price_usd: double (nullable = true)
 |-- price_original: double (nullable = true)
 |-- price_original_currency: string (nullable = true)
 |-- cpu: string (nullable = true)
 |-- ram_gb: double (nullable = true)
 |-- storage: string (nullable = true)
 |-- gpu: string (nullable = true)
 |-- display: string (nullable = true)
 |-- battery: string (nullable = true)
 |-- category: string (nullable = true)
 |-- document_text: string (nullable = true)
 |-- has_review_text: string (nullable = true)
 |-- source_dataset: string (nullable = true)
 |-- source_collection_date: string (nullable = true)
 |-- source_row_id: string (nullable = true)
 |-- has_page_provenance: string (nullable = true)
 |-- ram_gb_is_imputed: boolean (nullable = true)
 |-- battery_is_imputed: boolean (nullable = true)
 |-- display_is_imputed: boolean (nullable = true)
 |-- cpu_is_imputed: boolean (nullable = 

In [33]:
CHUNK_WORDS = 120
CHUNK_OVERLAP = 20


def chunk_partition(rows):

    for row in rows:

        text = row["document_text"] or ""

        words = text.split()

        if not words:
            continue

        start = 0
        idx = 0
        n = len(words)

        while start < n:

            end = min(
                start + CHUNK_WORDS,
                n
            )

            chunk_text = " ".join(
                words[start:end]
            )

            yield (
                row["row_uid"],
                idx,
                chunk_text
            )

            if end == n:
                break

            start = end - CHUNK_OVERLAP
            idx += 1


chunk_schema = T.StructType([
    T.StructField(
        "row_uid",
        T.StringType(),
        False
    ),
    T.StructField(
        "chunk_idx",
        T.IntegerType(),
        False
    ),
    T.StructField(
        "chunk_text",
        T.StringType(),
        False
    )
])


chunks_rdd = (
    final_spark
    .select(
        "row_uid",
        "document_text"
    )
    .rdd
    .mapPartitions(chunk_partition)
)


chunks_df = spark.createDataFrame(
    chunks_rdd,
    schema=chunk_schema
)

print(
    "Total chunks:",
    chunks_df.count()
)

Total chunks: 4171


In [34]:
metadata_cols = [
    "row_uid",
    "title",
    "price_usd",
    "price_original",
    "price_original_currency",
    "cpu",
    "ram_gb",
    "storage",
    "gpu",
    "display",
    "battery",
    "category",
    "has_review_text",
    "source_dataset",
    "source_collection_date",
    "source_row_id",
    "has_page_provenance"
]


enriched = (
    chunks_df
    .join(
        final_spark.select(
            *metadata_cols
        ),
        on="row_uid",
        how="left"
    )
)

print(
    "Enriched chunks:",
    enriched.count()
)

Enriched chunks: 4171


In [35]:
from pyspark.sql.functions import pandas_udf


EMBED_MODEL_NAME = "all-MiniLM-L6-v2"


@pandas_udf(T.ArrayType(T.FloatType()))
def embed_batch(
    texts: pd.Series
) -> pd.Series:

    from sentence_transformers import SentenceTransformer

    global _model

    if "_model" not in globals():

        _model = SentenceTransformer(
            EMBED_MODEL_NAME
        )

    vecs = _model.encode(
        texts.tolist(),
        batch_size=64,
        show_progress_bar=False
    )

    return pd.Series(
        [
            v.tolist()
            for v in vecs
        ]
    )


embedded = (
    enriched
    .coalesce(4)
    .withColumn(
        "embedding",
        embed_batch(
            F.col("chunk_text")
        )
    )
)

embedded.select(
    "row_uid",
    "chunk_idx",
    "chunk_text",
    "embedding"
).show(
    3,
    truncate=60
)

+--------------------------------+---------+------------------------------------------------------------+------------------------------------------------------------+
|                         row_uid|chunk_idx|                                                  chunk_text|                                                   embedding|
+--------------------------------+---------+------------------------------------------------------------+------------------------------------------------------------+
|558389f6752bb2a5f024a76c8b26a6ed|        0|Product: ASUS Vivobook S 16" 3K OLED Intel Core Ultra 7 2...|[-0.07516628, -0.0036861245, -0.07122589, -0.011114168, 0...|
|558389f6752bb2a5f024a76c8b26a6ed|        1|13, Intel AI Boost NPU up to 13TOPS; Screen Size: 16"; To...|[-0.0380943, -0.017700111, 0.020603169, -0.030108942, 0.0...|
|558389f6752bb2a5f024a76c8b26a6ed|        2|40Gbps); HDMI: 1 x HDMI 2.1; Audio Ports: 1 x 3.5mm Combo...|[-0.008703979, -0.014219196, -0.008708181, -0.014658579, ...

In [36]:
PARQUET_PATH = "/content/laptop_chunks_embeddings.parquet"

(
    embedded
    .withColumn(
        "chunk_id",
        F.concat_ws(
            "_",
            F.col("row_uid"),
            F.col("chunk_idx")
        )
    )
    .write
    .mode("overwrite")
    .parquet(
        PARQUET_PATH
    )
)

print(
    "Wrote:",
    PARQUET_PATH
)

Wrote: /content/laptop_chunks_embeddings.parquet


In [37]:
import pandas as pd


embedded_pdf = pd.read_parquet(
    PARQUET_PATH
)


embedded_pdf["lineage"] = (
    "chunk:"
    + embedded_pdf["chunk_id"].astype(str)
    + " <- device:"
    + embedded_pdf["row_uid"].astype(str)
    + " <- source_row:"
    + embedded_pdf["source_row_id"].astype(str)
    + " <- dataset:"
    + embedded_pdf["source_dataset"].astype(str)
)


FINAL_PARQUET = "/content/laptop_chunks_embeddings_with_lineage.parquet"


embedded_pdf.to_parquet(
    FINAL_PARQUET,
    index=False
)


print(
    "Saved:",
    FINAL_PARQUET
)

print(
    embedded_pdf[
        [
            "chunk_id",
            "row_uid",
            "source_row_id",
            "source_dataset",
            "lineage"
        ]
    ]
    .head(3)
    .to_string()
)

Saved: /content/laptop_chunks_embeddings_with_lineage.parquet
                             chunk_id                           row_uid                                                                                                                                                           source_row_id source_dataset                                                                                                                                                                                                                                                                                            lineage
0  558389f6752bb2a5f024a76c8b26a6ed_0  558389f6752bb2a5f024a76c8b26a6ed  https://www.newegg.com/asus-vivobook-16-2880x1800-oled-intel-core-ultra-7-255h-intel-arc-graphics-gpu-32-gb-memory-1-tb-pcie-g4-ssd-no-hdd-hdd-black/p/N82E16834236727  newegg_scrape  chunk:558389f6752bb2a5f024a76c8b26a6ed_0 <- device:558389f6752bb2a5f024a76c8b26a6ed <- source_row:https://www.newegg.com/asus-vivobook-1

In [38]:
print("=" * 70)
print("FINAL PIPELINE STATISTICS")
print("=" * 70)

print(
    "Original rows:",
    original_count
)

print(
    "Rows after deduplication:",
    deduped_count
)

print(
    "Duplicate rows removed:",
    original_count - deduped_count
)

print(
    "Total chunks:",
    chunks_df.count()
)

print(
    "Average chunks per laptop:",
    chunks_df.count() / deduped_count
)

embedding_dim = len(
    embedded
    .select("embedding")
    .first()["embedding"]
)

print(
    "Embedding dimension:",
    embedding_dim
)

print(
    "\nFinal product CSV:",
    FINAL_CSV
)

print(
    "Final Parquet:",
    FINAL_PARQUET
)

FINAL PIPELINE STATISTICS
Original rows: 2955
Rows after deduplication: 1960
Duplicate rows removed: 995
Total chunks: 4171
Average chunks per laptop: 2.128061224489796
Embedding dimension: 384

Final product CSV: /content/final_clean_filled_laptops.csv
Final Parquet: /content/laptop_chunks_embeddings_with_lineage.parquet
